# Telco Customer Churn Prediction

This notebook prepares the customer-level dataset, trains a Random Forest classifier, evaluates performance on a stratified holdout set, examines feature importance, and exports customer-level churn predictions for Power BI.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)

In [ ]:
# Run the notebook from the repository root.
file_path = 'data/processed/telco_customer_detail.csv'
data = pd.read_csv(file_path, encoding='utf-8-sig')

# Remove hidden BOM characters and whitespace from headers.
data.columns = (
    data.columns
    .str.replace('ï»¿', '', regex=False)
    .str.replace('\ufeff', '', regex=False)
    .str.strip()
)

data.head()

In [ ]:
print('Dataset shape:', data.shape)
print(data['Churn'].value_counts())
print(data.isna().sum().sort_values(ascending=False).head())

In [ ]:
df = data.copy()

# IDs identify customers but should not be treated as predictive signals.
df = df.drop(columns=['customerID'])

# TotalCharges may contain blank strings in the original source.
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# Encode target.
df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1})

X = df.drop(columns=['Churn'])
y = df['Churn']

# One-hot encoding avoids imposing false ordinal relationships on categories.
X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print('Training rows:', X_train.shape[0])
print('Testing rows:', X_test.shape[0])
print('Encoded features:', X.shape[1])

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
)
rf_model.fit(X_train, y_train)

In [ ]:
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]

metrics = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1 score': f1_score(y_test, y_pred),
    'ROC AUC': roc_auc_score(y_test, y_prob),
}

print(pd.Series(metrics).round(4))
print('
Confusion matrix:')
print(confusion_matrix(y_test, y_pred))
print('
Classification report:')
print(classification_report(y_test, y_pred))

In [ ]:
feature_importance = (
    pd.DataFrame({
        'Feature': X.columns,
        'Importance': rf_model.feature_importances_,
    })
    .sort_values('Importance', ascending=False)
)

feature_importance.head(15)

In [ ]:
top_features = feature_importance.head(15).sort_values('Importance')

plt.figure(figsize=(10, 7))
plt.barh(top_features['Feature'], top_features['Importance'])
plt.title('Top 15 Random Forest Feature Importances')
plt.xlabel('Relative importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

In [ ]:
# Score the full customer dataset for dashboard use.
scoring_data = data.copy()
customer_ids = scoring_data['customerID'].copy()
actual_churn = scoring_data['Churn'].copy()

scoring_features = scoring_data.drop(columns=['customerID', 'Churn'])
scoring_features['TotalCharges'] = pd.to_numeric(
    scoring_features['TotalCharges'], errors='coerce'
)
scoring_features['TotalCharges'] = scoring_features['TotalCharges'].fillna(
    scoring_features['TotalCharges'].median()
)
scoring_features = pd.get_dummies(scoring_features, drop_first=True)
scoring_features = scoring_features.reindex(columns=X.columns, fill_value=0)

scoring_data['PredictedChurn'] = rf_model.predict(scoring_features)
scoring_data['PredictedChurnLabel'] = scoring_data['PredictedChurn'].map({0: 'No', 1: 'Yes'})
scoring_data['ChurnProbability'] = rf_model.predict_proba(scoring_features)[:, 1]

output_path = 'outputs/telco_churn_predictions.csv'
scoring_data.to_csv(output_path, index=False)
joblib.dump(rf_model, 'outputs/random_forest_churn_model.joblib')

print(f'Predictions saved to {output_path}')
scoring_data[['customerID', 'PredictedChurnLabel', 'ChurnProbability']].head()